# UAPP — S669 external validation (D5 ensemble)

Train a 5-member D5 ensemble on T2837, evaluate on S669 (Pancotti et al. 2022). This is the cleanest available test of generalisation for the campaign's production model.

**What you need in Drive (`/content/drive/MyDrive/uapp_cache/`) before starting:**
- `t2837_embeddings_v2_650m.pt`  (~26 MB, ESM2-650M cache)
- `t2837_metadata.csv`
- `t2837_bio_features_650m_extended.pt`  (k=13 D5 features)

If any are missing, rebuild via the recipe in `REPORT.md` §9 before continuing.

**Path:** D5 only (k=13: RSA + chemistry + sequence-derived structural). No DSSP / AlphaFold downloads needed. D5 had the best Spearman in the campaign (0.348 vs D6's 0.331).

## 1. Environment

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf uapp
!git clone https://github.com/RoselindSi/uapp.git
%cd /content/uapp
!git log --oneline -5

In [ ]:
!pip install -q transformers torch numpy pandas tqdm scipy biopython

## 2. Restore T2837 caches from Drive

If any are missing, **stop here** and rebuild — the S669 eval needs them.

In [ ]:
import os, shutil
os.makedirs('cache', exist_ok=True)

needed = [
    't2837_embeddings_v2_650m.pt',
    't2837_metadata.csv',
    't2837_bio_features_650m_extended.pt',
]
missing = []
for f in needed:
    src = f'/content/drive/MyDrive/uapp_cache/{f}'
    dst = f'cache/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✓ {f}  ({os.path.getsize(dst)/1e6:.1f} MB)')
    else:
        print(f'✗ MISSING: {f}')
        missing.append(f)

assert not missing, f'Missing required caches: {missing}.  Rebuild before continuing.'

## 3. Get the S669 dataset

Authoritative source: Pancotti et al. 2022 on Zenodo (record 7568094). The zip contains `S669/S669.csv` plus WT and mutant PDB structures — we only need the CSV for D5.

In [ ]:
!mkdir -p data/s669
!wget -q -O data/s669/S669.zip https://zenodo.org/records/7568094/files/S669.zip
!unzip -o -q data/s669/S669.zip -d data/s669/
!ls data/s669/S669/
!head -3 data/s669/S669/S669.csv
!wc -l data/s669/S669/S669.csv

In [ ]:
# Manual-upload fallback (skip if previous cell worked)
# from google.colab import files
# uploaded = files.upload()
# import shutil, os
# fname = list(uploaded.keys())[0]
# os.makedirs('data/s669/S669', exist_ok=True)
# shutil.move(fname, 'data/s669/S669/S669.csv')
# !head -3 data/s669/S669/S669.csv && wc -l data/s669/S669/S669.csv

## 4. Extract WT sequences from PDBs, then convert S669 → T2837 metadata

The Zenodo S669 release has no inline sequence column — sequences come from the bundled WT PDB files (`data/s669/S669/pdbs/`). We extract one WT sequence per unique `Protein` value (using the per-row `WT_PDB` + `Chain` columns) into a FASTA, then hand it to script 17.

In [ ]:
# 4a. Extract WT sequences from the bundled PDBs → FASTA keyed by `Protein`
from pathlib import Path
from Bio.PDB import PDBParser
import pandas as pd

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLU':'E','GLN':'Q',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}

df = pd.read_csv('data/s669/S669/S669.csv')
print('First rows:')
print(df[['Protein', 'WT_PDB', 'Chain', 'Seq_Mut', 'Experimental_DDG_dir']].head())

pdb_dir = Path('data/s669/S669/pdbs')
parser = PDBParser(QUIET=True)

fasta_lines, seen, n_missing = [], set(), 0
for _, row in df.iterrows():
    prot = str(row['Protein'])
    if prot in seen:
        continue
    seen.add(prot)
    pdb_name = str(row['WT_PDB']).strip()
    chain_id = str(row['Chain']).strip()
    candidates = [pdb_dir / pdb_name, pdb_dir / f'{pdb_name}.pdb',
                  pdb_dir / pdb_name.lower(), pdb_dir / f'{pdb_name.lower()}.pdb']
    pdb_path = next((c for c in candidates if c.exists()), None)
    if pdb_path is None:
        n_missing += 1; continue
    structure = parser.get_structure(prot, pdb_path)
    seq = ''
    for model in structure:
        for chain in model:
            if chain.id != chain_id:
                continue
            for residue in chain:
                if residue.id[0] != ' ':
                    continue
                seq += THREE_TO_ONE.get(residue.resname, 'X')
            break
        break
    if seq:
        fasta_lines.append(f'>{prot}\n{seq}\n')

Path('data/s669/wildtypes.fasta').write_text(''.join(fasta_lines))
print(f'\nWrote {len(fasta_lines)} sequences to data/s669/wildtypes.fasta '
      f'({n_missing} proteins had no PDB file)')
!head -4 data/s669/wildtypes.fasta

In [ ]:
# 4b. Convert S669 → T2837 metadata format using the FASTA we just built
!python scripts/17_s669_to_t2837_format.py \
    --s669-csv data/s669/S669/S669.csv \
    --fasta    data/s669/wildtypes.fasta \
    --protein-col Protein \
    --mut-col     Seq_Mut \
    --ddg-col     Experimental_DDG_dir \
    --out cache/s669_metadata.csv

## 5. Cache S669 ESM2-650M embeddings

~3 min on a T4. Script 01 enforces `max(1, ...)` per split, so a couple of rows may end up in `train` or `val` instead of `test` — script 18 picks the largest split automatically and warns.

## 6. Build S669 D5 bio features (k=13)

Script 06 fits its standardiser on the cache's `train` split, but S669 is test-only — and we want S669 features standardised with **T2837's** train statistics anyway, since that's what the trained model expects. So we bypass script 06 with an inline build that reuses `batch_features_extended` from `uapp.mutation_features` and applies T2837's `mu`/`sd` (stored in the T2837 bio-features file's `meta`).

In [ ]:
import sys, numpy as np, pandas as pd, torch
sys.path.insert(0, '/content/uapp')

from uapp.data import load_cached_embeddings
from uapp.mutation_features import batch_features_extended

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLU':'E','GLN':'Q',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}
def to1(x):
    x = str(x).strip().upper()
    return THREE_TO_ONE.get(x, x if len(x) == 1 else 'X')

# 1. Load S669 metadata + cache (test-only)
md = pd.read_csv('cache/s669_metadata_processed.csv')
md['split'] = md['split'].astype(str).str.lower()
splits, _ = load_cached_embeddings('cache/s669_embeddings_650m.pt')

# Find the non-empty split (script 01's max(1,...) clamp may stash it under any key)
nonempty = [s for s, (X, _) in splits.items() if X.shape[0] > 0]
assert len(nonempty) == 1, f'expected one non-empty split, got {nonempty}'
split_key = nonempty[0]
n_cache = splits[split_key][0].shape[0]
g = md[md['split'] == split_key].reset_index(drop=True)
assert len(g) == n_cache, f'metadata/cache row mismatch: {len(g)} vs {n_cache}'

# 2. Build raw extended features (k=13)
raw = batch_features_extended(
    [to1(x) for x in g['wtAA']],
    [to1(x) for x in g['mutAA']],
    g['rel_rsa'].astype(float).to_numpy(),
    sequences=g['sequence'].astype(str).tolist(),
    mut_indices=g['mut_idx'].astype(int).to_numpy(),
    include_indicators=False,
)
print(f'Raw S669 features: {raw.shape}')

# 3. Load T2837's mu/sd and standardise S669 with them
t_bio = torch.load('cache/t2837_bio_features_650m_extended.pt',
                   map_location='cpu', weights_only=False)
mu = np.asarray(t_bio['meta']['mu'], dtype=np.float64)
sd = np.asarray(t_bio['meta']['sd'], dtype=np.float64)
sd = np.where(sd < 1e-6, 1.0, sd)
print(f"T2837 mu/sd: k={len(mu)}  mu[:3]={mu[:3]}  sd[:3]={sd[:3]}")
assert raw.shape[1] == len(mu), 'feature-dim mismatch vs T2837'
standardised = ((raw - mu) / sd).astype(np.float32)

# 4. Save in the payload shape script 18 expects
payload = {
    'train': {'feats': torch.zeros(0, len(mu))},
    'val':   {'feats': torch.zeros(0, len(mu))},
    'test':  {'feats': torch.zeros(0, len(mu))},
}
payload[split_key] = {'feats': torch.from_numpy(standardised)}
payload['meta'] = {
    'feature_names':      t_bio['meta']['feature_names'],
    'k':                  int(len(mu)),
    'mu':                 mu.tolist(),
    'sd':                 sd.tolist(),
    'include_indicators': False,
    'include_extended':   True,
    'source_metadata':    'cache/s669_metadata_processed.csv',
    'source_embeddings':  'cache/s669_embeddings_650m.pt',
    'standardiser_from':  'cache/t2837_bio_features_650m_extended.pt',
}
torch.save(payload, 'cache/s669_bio_features_650m_extended.pt')
print(f"Saved cache/s669_bio_features_650m_extended.pt  "
      f"(test split: {standardised.shape}, standardised with T2837 stats)")

In [ ]:
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/s669_metadata_processed.csv \
    --embeddings   cache/s669_embeddings_650m.pt \
    --out          cache/s669_bio_features_650m_extended.pt \
    --include-extended

## 7. Train D5 ensemble on T2837, evaluate on S669

~5 min on a T4. Trains 5 D5 heads on T2837 train, early-stops on T2837 val, predicts on:
- T2837 test — sanity check, should reproduce campaign numbers (RMSE ≈ 1.50, Spearman ≈ 0.348 within ± 0.07).
- S669 — the headline external-validation result.

In [ ]:
!python scripts/18_evaluate_on_s669.py \
    --t2837-emb cache/t2837_embeddings_v2_650m.pt \
    --t2837-bio cache/t2837_bio_features_650m_extended.pt \
    --s669-emb  cache/s669_embeddings_650m.pt \
    --s669-bio  cache/s669_bio_features_650m_extended.pt \
    --out       outputs/s669_eval_d5 \
    --ablation D5 --members 5 --device cuda

## 8. Save outputs to Drive

In [ ]:
!cp -r outputs/s669_eval_d5 /content/drive/MyDrive/uapp_cache/
!cp cache/s669_metadata.csv \
    cache/s669_metadata_processed.csv \
    cache/s669_embeddings_650m.pt \
    cache/s669_bio_features_650m_extended.pt \
    /content/drive/MyDrive/uapp_cache/
print('saved to Drive')

## 9. Inspect the result

In [ ]:
import json, pathlib
s = json.loads(pathlib.Path('outputs/s669_eval_d5/ensemble_summary.json').read_text())

print('=' * 70)
print('T2837 test ensemble  (sanity check vs campaign numbers)')
print('=' * 70)
for k in ('n_test', 'rmse', 'mae', 'nll', 'ice', 'spearman',
         'cov@0.90', 'cov@0.95', 'top0.20'):
    print(f'  {k:<12} = {s["t2837_test"]["ensemble"][k]}')

print()
print('=' * 70)
print('S669 ensemble  (headline external validation)')
print('=' * 70)
for k in ('n_test', 'rmse', 'mae', 'nll', 'ice', 'spearman',
         'cov@0.90', 'cov@0.95', 'top0.20'):
    print(f'  {k:<12} = {s["s669"]["ensemble"][k]}')

print()
print('=' * 70)
print('Cross-dataset Δ (S669 − T2837)')
print('=' * 70)
te = s['t2837_test']['ensemble']; se = s['s669']['ensemble']
for k in ('rmse', 'nll', 'ice', 'spearman'):
    print(f'  Δ {k:<10} = {se[k] - te[k]:+.4f}')

## Decision rule

| S669 ensemble Spearman | Verdict | Action |
|---|---|---|
| ≥ 0.30 | Strong generalisation | Add S669 row to `REPORT.md` deliverable table; unstrike Limitation #5 |
| 0.20–0.30 | Real but degraded | Document the drop honestly; still a publishable finding |
| < 0.20 | Generalisation fails | Document as negative result; investigate domain shift |

T2837 test sanity row should land within ~0.05 of (RMSE 1.50, Spearman 0.348). If it's wildly off, something's wrong with the cache or feature alignment — stop and check before trusting the S669 numbers.